# Multimodal Search Pipeline — AMI Meeting Corpus ES2008a

I built this pipeline to explore how the audio and visual content of a real meeting can be unified into a single semantic search system. The pipeline ingests a raw meeting recording (WAV) and the accompanying presentation slides (JPGs), transcribes and OCRs them locally, embeds everything into a shared vector space, and exposes a cross-modal search interface that lets you find relevant moments regardless of whether the answer is spoken or shown on a slide.

The sample dataset is meeting **ES2008a** from the [AMI Meeting Corpus](https://groups.inf.ed.ac.uk/ami/corpus/) — a 30-minute scenario meeting about product design, chosen because it is publicly available, compact enough to process in minutes on a laptop, and contains both spoken discussion and projected slides.

## Pipeline at a glance

```
WAV file  ──►  Whisper (ASR)  ──►  Transcript text
                                         │
Slide JPGs ──►  Tesseract (OCR) ──►  Slide text
                                         │
                      ┌──────────────────┘
                      ▼
             sentence-transformers          all-MiniLM-L6-v2
             (384-d embeddings)
                      │
           ┌──────────┴──────────┐
           ▼                     ▼
   ChromaDB collection    ChromaDB collection
   transcript_chunks        slides_ocr
           └──────────┬──────────┘
                      ▼
            Cross-modal semantic search
```

## Setup

I import every library up-front so missing dependencies surface immediately. The `step_times` dictionary collects wall-clock timings for the pipeline summary table in the Evaluation section.

In [ ]:
import os
import sys
import glob
import time
import random
import platform
import warnings
from pathlib import Path
import xml.etree.ElementTree as ET

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import whisper
import pytesseract
from PIL import Image
from sentence_transformers import SentenceTransformer
import chromadb
import jiwer

# Windows: point pytesseract at the default Tesseract install location.
# Adjust the path if you installed Tesseract elsewhere.
if platform.system() == "Windows":
    pytesseract.pytesseract.tesseract_cmd = (
        os.environ.get("TESSERACT_CMD",
                       r"C:\Program Files\Tesseract-OCR\tesseract.exe")
    )

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR       = Path("data")
AUDIO_DIR      = DATA_DIR / "raw" / "audio"
SLIDES_DIR     = DATA_DIR / "raw" / "slides"
TRANSCRIPT_DIR = DATA_DIR / "raw" / "transcripts"
CHROMA_DIR     = Path("chroma_db")

for _d in [AUDIO_DIR, SLIDES_DIR, TRANSCRIPT_DIR, CHROMA_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────
MEETING_ID    = "ES2008"
MEETING_PART  = "a"
CHUNK_SIZE    = 500
OVERLAP       = 50
WHISPER_MODEL = "base"
EMBED_MODEL   = "all-MiniLM-L6-v2"

# ── Timing registry ────────────────────────────────────────────────────────
step_times: dict = {}

print(f"Python {sys.version.split()[0]}  |  Platform: {platform.system()}")
print("Setup complete.")

---
## Step 1 — Audio Transcription

I use OpenAI Whisper to transcribe the raw meeting recording. Whisper is an encoder-decoder transformer trained on 680k hours of multilingual speech; it runs fully offline once the model weights are downloaded. I default to `whisper-base` (74 M parameters) as a practical balance between speed and accuracy — the Evaluation section benchmarks it against `tiny` and `small`.

The output is stored in a tidy DataFrame with one row per meeting segment so it is easy to join against other metadata tables later.

In [ ]:
audio_path = AUDIO_DIR / "ES2008a.Mix-Headset.wav"

if not audio_path.exists():
    raise FileNotFoundError(
        f"Audio file not found: {audio_path}\n"
        "Run  python download_data.py  to fetch the AMI corpus data."
    )

print(f"Loading Whisper model: {WHISPER_MODEL} …")
_t0 = time.time()
whisper_model = whisper.load_model(WHISPER_MODEL)

print(f"Transcribing {audio_path.name} …")
result         = whisper_model.transcribe(str(audio_path))
step_times["Transcription"] = time.time() - _t0

transcript_text = result["text"].strip()
audio_duration  = result.get("duration", 0.0)

transcription_df = pd.DataFrame([{
    "meeting_id"      : MEETING_ID,
    "meeting_part"    : MEETING_PART,
    "audio_file_path" : str(audio_path),
    "transcript_text" : transcript_text,
    "audio_duration"  : round(audio_duration, 2),
}])

print(f"\nDuration  : {audio_duration:.1f} s")
print(f"Characters: {len(transcript_text):,}")
print(f"Time      : {step_times['Transcription']:.1f} s")
print(f"\nFirst 100 chars of transcript:")
print(transcript_text[:100])
print()
display(transcription_df[["meeting_id", "meeting_part", "audio_duration"]])

---
## Step 2 — Slide OCR

I extract text from each slide JPEG using Tesseract with the `--psm 1` page-segmentation mode (full automatic layout analysis with orientation detection). This handles multi-column layouts and mixed text blocks better than the default single-column mode, which is important for slides that contain bullet lists alongside diagrams.

Each slide becomes one row in `slides_df`. Slides with no detectable text (e.g., purely graphical) still get a row — their `extracted_text` will be an empty string.

In [ ]:
slide_files = sorted(glob.glob(str(SLIDES_DIR / "*.jpg")))

if not slide_files:
    raise FileNotFoundError(
        f"No JPG slides found in {SLIDES_DIR}\n"
        "Run  python download_data.py  to fetch the AMI corpus slides."
    )

_t0 = time.time()
ocr_records = []
for slide_path in slide_files:
    img  = Image.open(slide_path).convert("RGB")
    text = pytesseract.image_to_string(img, config="--oem 3 --psm 1").strip()
    ocr_records.append({
        "meeting_id"     : MEETING_ID,
        "meeting_part"   : MEETING_PART,
        "slide_filename" : Path(slide_path).name,
        "slide_file_path": slide_path,
        "extracted_text" : text,
    })
    preview = text[:100].replace("\n", " ")
    print(f"  {Path(slide_path).name:<40s}  →  {preview!r}")

step_times["OCR"] = time.time() - _t0
slides_df = pd.DataFrame(ocr_records)

print(f"\n{len(slides_df)} slides processed in {step_times['OCR']:.1f} s")
display(slides_df[["slide_filename", "meeting_id", "meeting_part"]].head(10))

---
## Step 3 — Text Chunking

A raw transcript is too long to embed as a single vector — semantic meaning gets diluted over thousands of tokens. I split the transcript into overlapping windows of **500 characters** with a **50-character overlap**. The overlap ensures that sentences that fall near a chunk boundary are captured in at least one chunk, preventing information loss at the seams.

Each chunk gets a deterministic ID in the format `{meeting_id}_{part}_{chunk_index}` so results from any downstream step can be traced back to their source position in the transcript.

In [ ]:
def chunk_text(text: str, meeting_id: str, part: str,
               chunk_size: int = 500, overlap: int = 50) -> list[dict]:
    chunks, start, idx = [], 0, 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append({
            "chunk_id"    : f"{meeting_id}_{part}_{idx}",
            "meeting_part": part,
            "text"        : text[start:end],
            "start_char"  : start,
            "end_char"    : end,
        })
        start += chunk_size - overlap
        idx   += 1
    return chunks


chunks = chunk_text(
    transcript_text, MEETING_ID, MEETING_PART,
    chunk_size=CHUNK_SIZE, overlap=OVERLAP,
)
chunks_df = pd.DataFrame(chunks)

avg_len = chunks_df["text"].str.len().mean()
print(f"Chunk count   : {len(chunks_df)}")
print(f"Average length: {avg_len:.1f} chars")

print("\n── 10 random chunks ──────────────────────────────────────────────")
sample = chunks_df.sample(min(10, len(chunks_df)), random_state=42)
for row in sample.itertuples():
    print(f"\n[{row.chunk_id}]")
    print(row.text)

---
## Step 4 — Generating Embeddings

### Why `all-MiniLM-L6-v2`?

I chose `all-MiniLM-L6-v2` from the `sentence-transformers` library for three reasons:

1. **Speed** — At 22 M parameters it is one of the fastest sentence encoders available. On CPU it processes hundreds of short passages in seconds, making it practical for local, offline use without a GPU.
2. **Dimensionality** — The 384-dimension output is compact: small enough for fast cosine-similarity search and low ChromaDB storage overhead, yet expressive enough to capture the nuanced semantic differences between meeting topics.
3. **General quality** — It was distilled from a much larger model using multi-task training on diverse corpora, so it generalises well to domain-specific text like meeting transcripts and slide content without any fine-tuning.

**Production upgrade path**: `all-mpnet-base-v2` is a drop-in replacement that produces meaningfully better retrieval quality at the cost of roughly 2× the latency. The *only* code change required is the model name string:

```python
embed_model = SentenceTransformer("all-mpnet-base-v2")
```

### ChromaDB storage

I store embeddings in two separate ChromaDB collections:

| Collection | Content | Documents |
|---|---|---|
| `transcript_chunks` | Overlapping 500-char transcript windows | One per chunk |
| `slides_ocr` | Full OCR text per slide | One per slide |

Using cosine similarity (`hnsw:space = cosine`) normalises for document length, which matters because slide text is typically much shorter than transcript chunks.

In [ ]:
_t0 = time.time()

print(f"Loading sentence-transformer: {EMBED_MODEL} …")
embed_model = SentenceTransformer(EMBED_MODEL)

# ── Embed transcript chunks ────────────────────────────────────────────────
chunk_texts = chunks_df["text"].tolist()
chunk_ids   = chunks_df["chunk_id"].tolist()
print(f"Embedding {len(chunk_texts)} transcript chunks …")
chunk_embeddings = embed_model.encode(
    chunk_texts, show_progress_bar=True, convert_to_numpy=True
)

# ── Embed slide OCR text ───────────────────────────────────────────────────
slide_texts = slides_df["extracted_text"].tolist()
slide_ids   = [
    fn.replace(".jpg", "") for fn in slides_df["slide_filename"].tolist()
]
print(f"Embedding {len(slide_texts)} slide texts …")
slide_embeddings = embed_model.encode(
    slide_texts, show_progress_bar=True, convert_to_numpy=True
)

step_times["Embedding"] = time.time() - _t0

# ── ChromaDB ───────────────────────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

for _col_name in ["transcript_chunks", "slides_ocr"]:
    try:
        chroma_client.delete_collection(_col_name)
    except Exception:
        pass

transcript_col = chroma_client.create_collection(
    "transcript_chunks",
    metadata={"hnsw:space": "cosine"},
)
slides_col = chroma_client.create_collection(
    "slides_ocr",
    metadata={"hnsw:space": "cosine"},
)

# Add transcript chunks in batches of 100
BATCH = 100
for i in range(0, len(chunk_ids), BATCH):
    batch_ids  = chunk_ids[i:i + BATCH]
    batch_embs = chunk_embeddings[i:i + BATCH].tolist()
    batch_docs = chunk_texts[i:i + BATCH]
    batch_meta = [{"meeting_part": MEETING_PART} for _ in batch_ids]
    transcript_col.add(
        ids=batch_ids, embeddings=batch_embs,
        documents=batch_docs, metadatas=batch_meta,
    )

slides_col.add(
    ids=slide_ids,
    embeddings=slide_embeddings.tolist(),
    documents=slide_texts,
    metadatas=[
        {"meeting_part": MEETING_PART,
         "slide_filename": slides_df.iloc[j]["slide_filename"]}
        for j in range(len(slide_ids))
    ],
)

print(f"\nChromaDB collections created:")
print(f"  transcript_chunks : {transcript_col.count()} documents")
print(f"  slides_ocr        : {slides_col.count()} documents")
print(f"  Embedding time    : {step_times['Embedding']:.1f} s")

---
## Step 5 — Semantic Search: Audio

I embed the query with the same model used to embed the corpus — this is the key invariant for semantic search to work. ChromaDB returns cosine distances (lower = more similar), which I convert to similarity scores (`1 - distance`) for readability.

Replace the placeholder query with any natural-language question about the meeting content.

In [ ]:
QUERY_AUDIO = "REPLACE WITH YOUR QUERY"

_t0 = time.time()
q_emb = embed_model.encode([QUERY_AUDIO], convert_to_numpy=True)
audio_results = transcript_col.query(
    query_embeddings=q_emb.tolist(),
    n_results=5,
    include=["documents", "metadatas", "distances"],
)
step_times["Search Audio"] = time.time() - _t0

print(f"Query: {QUERY_AUDIO!r}")
print(f"Search time: {step_times['Search Audio'] * 1000:.1f} ms\n")

for rank, (cid, doc, meta, dist) in enumerate(
    zip(
        audio_results["ids"][0],
        audio_results["documents"][0],
        audio_results["metadatas"][0],
        audio_results["distances"][0],
    ),
    start=1,
):
    sim = 1 - dist
    print(f"Rank {rank} | chunk_id={cid} | meeting_part={meta['meeting_part']} | similarity={sim:.4f}")
    print(f"  {doc[:500]}")
    print()

---
## Step 6 — Semantic Search: Slides

The same search pattern applies to the slides collection. Because slide text is typically sparser than transcript text, similarity scores will often be lower — this is expected and does not indicate worse retrieval quality.

In [ ]:
QUERY_SLIDES = "REPLACE WITH YOUR QUERY"

_t0 = time.time()
q_emb_slides = embed_model.encode([QUERY_SLIDES], convert_to_numpy=True)
slides_results = slides_col.query(
    query_embeddings=q_emb_slides.tolist(),
    n_results=5,
    include=["documents", "metadatas", "distances"],
)
step_times["Search Slides"] = time.time() - _t0

print(f"Query: {QUERY_SLIDES!r}")
print(f"Search time: {step_times['Search Slides'] * 1000:.1f} ms\n")

for rank, (sid, doc, meta, dist) in enumerate(
    zip(
        slides_results["ids"][0],
        slides_results["documents"][0],
        slides_results["metadatas"][0],
        slides_results["distances"][0],
    ),
    start=1,
):
    sim = 1 - dist
    fname = meta.get("slide_filename", sid)
    print(f"Rank {rank} | slide_filename={fname} | meeting_part={meta['meeting_part']} | similarity={sim:.4f}")
    print(f"  {doc[:500]}")
    print()

---
## Step 7 — Cross-Modal Search

The most compelling feature of a multimodal pipeline is the ability to ask a single question and get answers from whichever modality is most relevant — without knowing in advance whether the answer was spoken or shown on screen.

I query both ChromaDB collections independently, tag each result with its modality (`AUDIO` or `SLIDE`), merge the lists, and re-sort by similarity score. This gives a globally ranked list of the most relevant meeting moments across both channels.

In [ ]:
QUERY_CROSS = "REPLACE WITH YOUR QUERY"

q_emb_cross = embed_model.encode([QUERY_CROSS], convert_to_numpy=True)

a_res = transcript_col.query(
    query_embeddings=q_emb_cross.tolist(), n_results=5,
    include=["documents", "metadatas", "distances"],
)
s_res = slides_col.query(
    query_embeddings=q_emb_cross.tolist(), n_results=5,
    include=["documents", "metadatas", "distances"],
)

combined = []
for cid, doc, meta, dist in zip(
    a_res["ids"][0], a_res["documents"][0],
    a_res["metadatas"][0], a_res["distances"][0],
):
    combined.append({"content_type": "AUDIO", "id": cid,
                     "text": doc, "meta": meta, "similarity": 1 - dist})

for sid, doc, meta, dist in zip(
    s_res["ids"][0], s_res["documents"][0],
    s_res["metadatas"][0], s_res["distances"][0],
):
    combined.append({"content_type": "SLIDE", "id": sid,
                     "text": doc, "meta": meta, "similarity": 1 - dist})

combined.sort(key=lambda x: x["similarity"], reverse=True)

print(f"Cross-modal query: {QUERY_CROSS!r}\n")
for rank, r in enumerate(combined, start=1):
    label = f"[{r['content_type']}]"
    print(f"Rank {rank:2d} | {label:7s} | {r['id']} | sim={r['similarity']:.4f}")
    print(f"  {r['text'][:500]}")
    print()

---
## Step 8 — Evaluation

A pipeline without evaluation is incomplete. I evaluate four aspects:

| Part | Metric | What it measures |
|---|---|---|
| A | Word Error Rate (WER) | Transcription accuracy vs. human reference |
| B | Precision@5 | Retrieval relevance for labelled queries |
| C | Pipeline timing | End-to-end latency per step |
| D | Chunk size ablation | How chunk size affects retrieval speed and quality |

### Part A — Transcription Quality (WER)

Word Error Rate measures how many words differ between the Whisper hypothesis and the AMI human-annotated reference transcript. The reference comes from the AMI NXT annotations — per-speaker `words.xml` files that I parse, sort by start-time, and concatenate into a single reference string.

I compare three Whisper model sizes to show the speed-accuracy tradeoff.

In [ ]:
def parse_ami_reference(transcript_dir: Path, meeting: str = "ES2008a") -> str | None:
    """
    Load the AMI reference transcript. Two formats are supported:
      1. Plain-text  ES2008a.reference.txt  (produced by download_data.py)
      2. NXT XML     ES2008a.*.words.xml    (manually extracted from the corpus)
    Returns None when neither is found.
    """
    # 1 — plain text (fast path, produced by download_data.py)
    txt_file = transcript_dir / f"{meeting}.reference.txt"
    if txt_file.exists() and txt_file.stat().st_size > 0:
        return txt_file.read_text(encoding="utf-8").strip()

    # 2 — NXT XML (manual download path)
    xml_files = sorted(transcript_dir.glob(f"{meeting}.*.words.xml"))
    if not xml_files:
        return None

    timed_words: list[tuple[float, str]] = []
    for xf in xml_files:
        try:
            tree = ET.parse(xf)
            for elem in tree.iter():
                tag = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
                if tag == "w" and elem.text:
                    word = elem.text.strip()
                    if word and word not in {
                        "<vocalsound>", "<nonvocalsound>",
                        "IGNORE_TIME_SEGMENT_IN_SCORING",
                    }:
                        t = float(elem.get("starttime", 0))
                        timed_words.append((t, word))
        except ET.ParseError:
            continue

    timed_words.sort(key=lambda x: x[0])
    return " ".join(w for _, w in timed_words)


ref_transcript = parse_ami_reference(TRANSCRIPT_DIR)
if ref_transcript is None:
    print("[INFO] Reference transcript not found — WER comparison is shown\n"
          "       against the whisper-base output itself (WER = 0) as a\n"
          "       placeholder. Run download_data.py to get a real reference.")
    ref_transcript = transcript_text
else:
    print(f"Reference transcript loaded: {len(ref_transcript.split()):,} words")

wer_rows = []
for model_name in ["tiny", "base", "small"]:
    print(f"\nLoading whisper-{model_name} …")
    _m  = whisper.load_model(model_name)
    _t0 = time.time()
    _r  = _m.transcribe(str(audio_path))
    elapsed = time.time() - _t0
    hyp = _r["text"].strip()
    wer = jiwer.wer(ref_transcript, hyp)
    wer_rows.append({
        "Model"              : f"whisper-{model_name}",
        "WER"                : round(wer, 4),
        "Transcription Time": f"{elapsed:.1f}s",
    })
    print(f"  whisper-{model_name}: WER = {wer:.4f}  |  time = {elapsed:.1f}s")

wer_df = pd.DataFrame(wer_rows)
print("\n── Transcription Quality Summary ─────────────────────────────────")
print(wer_df.to_string(index=False))

### Part B — Search Quality (Precision@5)

Precision@5 answers: *of the top 5 retrieved chunks, how many were actually relevant?* To use this, I need a labelled test set — queries paired with the chunk IDs that a human would consider correct answers. I provide the framework with 5 placeholder queries; fill them in after exploring the transcript in Step 3.

Chunk IDs follow the format `ES2008_a_{index}` and can be read from the output of Step 3.

In [ ]:
def precision_at_5(retrieved_chunks: list[dict], relevant_chunks: list[str]) -> float:
    retrieved_top5 = [r["chunk_id"] for r in retrieved_chunks[:5]]
    hits = len(set(retrieved_top5) & set(relevant_chunks))
    return hits / 5


# ── Fill in your labelled queries below ────────────────────────────────────
test_set = [
    {"query": "", "relevant_chunks": []},
    {"query": "", "relevant_chunks": []},
    {"query": "", "relevant_chunks": []},
    {"query": "", "relevant_chunks": []},
    {"query": "", "relevant_chunks": []},
]

print("── Precision@5 Evaluation ─────────────────────────────────────────")
p5_results = []
all_empty = all(not item["query"] for item in test_set)

if all_empty:
    print("(test_set is empty — fill in queries and relevant_chunks to evaluate)")
else:
    for item in test_set:
        if not item["query"]:
            continue
        q_emb_p5 = embed_model.encode([item["query"]], convert_to_numpy=True)
        res = transcript_col.query(
            query_embeddings=q_emb_p5.tolist(),
            n_results=5,
            include=["documents", "distances"],
        )
        retrieved = [{"chunk_id": cid} for cid in res["ids"][0]]
        p5 = precision_at_5(retrieved, item["relevant_chunks"])
        p5_results.append({"Query": item["query"][:55], "Precision@5": p5})
        print(f"  {item['query'][:55]:55s}  →  P@5 = {p5:.2f}")

    if p5_results:
        avg = sum(r["Precision@5"] for r in p5_results) / len(p5_results)
        print(f"\n  Mean Precision@5 across {len(p5_results)} queries: {avg:.2f}")

### Part C — Pipeline Timing

I collected timestamps at each step using Python's `time` module. The summary below shows the wall-clock cost of each stage, which is useful for identifying bottlenecks and planning production scaling decisions.

In [ ]:
timing_rows = [
    {"Step": step, "Time (seconds)": f"{elapsed:.2f}"}
    for step, elapsed in step_times.items()
]
timing_df = pd.DataFrame(timing_rows)

print("── Pipeline Timing Summary ────────────────────────────────────────")
print(timing_df.to_string(index=False))

total = sum(step_times.values())
print(f"\n  Total pipeline time: {total:.1f}s")

### Part D — Chunk Size Ablation

Chunk size is one of the most impactful hyperparameters in a retrieval pipeline. Smaller chunks retrieve more precise passages but may miss context; larger chunks retain context but dilute the query signal. I re-run chunking and search at three sizes — **250**, **500**, and **1000** characters — and measure search latency.

Each experiment uses an ephemeral (in-memory) ChromaDB collection so the persistent store is not affected. The Precision@5 column requires a filled `test_set` from Part B.

In [ ]:
def run_chunk_size_experiment(
    sizes: list[int],
    embed_model: SentenceTransformer,
    transcript: str,
    meeting_id: str,
    part: str,
    test_queries: list[dict] | None = None,
) -> pd.DataFrame:
    rows = []
    probe_query = "project goals and team responsibilities"

    for cs in sizes:
        exp_chunks = chunk_text(transcript, meeting_id, part,
                                chunk_size=cs, overlap=cs // 10)
        exp_texts  = [c["text"] for c in exp_chunks]
        exp_ids    = [c["chunk_id"] for c in exp_chunks]

        exp_client = chromadb.EphemeralClient()
        try:
            exp_client.delete_collection("exp")
        except Exception:
            pass
        exp_col = exp_client.create_collection(
            "exp", metadata={"hnsw:space": "cosine"}
        )
        exp_embs = embed_model.encode(
            exp_texts, convert_to_numpy=True, show_progress_bar=False
        )
        exp_col.add(ids=exp_ids, embeddings=exp_embs.tolist(), documents=exp_texts)

        # Measure search latency (average of 3 runs)
        q_emb = embed_model.encode([probe_query], convert_to_numpy=True)
        latencies = []
        for _ in range(3):
            _t0 = time.time()
            exp_col.query(
                query_embeddings=q_emb.tolist(), n_results=5,
                include=["documents", "distances"],
            )
            latencies.append((time.time() - _t0) * 1000)
        avg_lat = sum(latencies) / len(latencies)

        # Precision@5 (requires labelled test_set)
        p5_scores = []
        if test_queries:
            for item in test_queries:
                if not item["query"] or not item["relevant_chunks"]:
                    continue
                q_e = embed_model.encode([item["query"]], convert_to_numpy=True)
                r   = exp_col.query(
                    query_embeddings=q_e.tolist(), n_results=5,
                    include=["documents"],
                )
                retr = [{"chunk_id": cid} for cid in r["ids"][0]]
                p5_scores.append(precision_at_5(retr, item["relevant_chunks"]))

        p5_str = f"{sum(p5_scores)/len(p5_scores):.2f}" if p5_scores else "— (fill test_set)"

        rows.append({
            "Chunk Size": cs,
            "Num Chunks": len(exp_chunks),
            "Precision@5": p5_str,
            "Search Latency (ms)": f"{avg_lat:.1f}",
        })
        print(f"  size={cs:4d}  chunks={len(exp_chunks):3d}  latency={avg_lat:.1f}ms  P@5={p5_str}")

    return pd.DataFrame(rows)


print("── Chunk Size Ablation ────────────────────────────────────────────")
chunk_exp_df = run_chunk_size_experiment(
    sizes=[250, 500, 1000],
    embed_model=embed_model,
    transcript=transcript_text,
    meeting_id=MEETING_ID,
    part=MEETING_PART,
    test_queries=test_set,
)
print()
print(chunk_exp_df.to_string(index=False))